# 05 · Deep remaining-useful-life model on NASA C-MAPSS (TPU, JAX/Keras)

The backend ships a gradient-boosted RUL model trained on CPU (`scripts/train_public_models.py`). This notebook trains a
**1D-CNN + GRU sequence model on Cloud TPU** with Keras 3 on the JAX backend — the path TPUs support best — and compares
it with that baseline on the official test sets. Same preprocessing (per-regime normalisation, RUL cap 125, windows of
30 cycles) via the repo's own code, so the comparison is fair.

**Runtime:** *TPU*. Works unchanged on GPU/CPU (slower). C-MAPSS is a NASA public-domain dataset (simulated engines).

In [ ]:
# Get the repo, regenerate the synthetic corpus (deterministic, ~5 s) and fetch the public datasets (checksummed)
!git clone -q https://github.com/<you>/aeronexus-ai.git /content/aeronexus-ai || true
%cd /content/aeronexus-ai/backend
!pip -q install -r requirements.txt
!python scripts/generate_data.py --train 4000
!python scripts/fetch_public_data.py

In [ ]:
!pip -q install -U keras jax
import os; os.environ["KERAS_BACKEND"] = "jax"
import jax, keras, numpy as np, json, sys
sys.path.insert(0, ".")
print(jax.devices())
from aeronexus.pdm.cmapss import load, load_rul, add_target, Normalizer, CAP, rmse, phm_score

In [ ]:
W = 30
def windows(z, sensors, with_y):
    X, y, units = [], [], []
    for u, g in z.groupby("unit"):
        a = g[sensors].to_numpy("float32")
        if len(a) < W:
            a = np.vstack([np.repeat(a[:1], W - len(a), 0), a])      # left-pad short histories
        if with_y:
            r = g["rul"].to_numpy("float32")
            r = np.r_[np.repeat(r[:1], max(0, W - len(r))), r]
            for i in range(W, len(a) + 1):
                X.append(a[i - W:i]); y.append(r[i - 1]); units.append(u)
        else:
            X.append(a[-W:]); units.append(u)
    return np.stack(X), (np.array(y) if with_y else None), np.array(units)

def model(n_feat):
    inp = keras.Input((W, n_feat))
    x = keras.layers.Conv1D(64, 5, padding="causal", activation="relu")(inp)
    x = keras.layers.Conv1D(64, 5, padding="causal", activation="relu")(x)
    x = keras.layers.GRU(64)(x)
    x = keras.layers.Dense(64, activation="relu")(x)
    out = keras.layers.Dense(1)(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return m

In [ ]:
results = {}
for subset in ["FD001", "FD002", "FD003", "FD004"]:
    tr = add_target(load(subset, "train")); norm = Normalizer.fit(tr)
    z = norm.transform(tr); z["rul"] = tr["rul"].values
    X, y, units = windows(z, norm.sensors, True)
    rng = np.random.default_rng(0); val_units = set(rng.choice(np.unique(units), size=max(10, len(np.unique(units)) // 5), replace=False))
    vm = np.isin(units, list(val_units))
    m = model(X.shape[-1])
    m.fit(X[~vm], y[~vm], validation_data=(X[vm], y[vm]), epochs=40, batch_size=512, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
    q = float(np.quantile(np.abs(m.predict(X[vm], verbose=0).ravel() - y[vm]), 0.9))   # conformal half-width
    te = load(subset, "test"); Xt, _, _ = windows(norm.transform(te), norm.sensors, False)
    pred = np.clip(m.predict(Xt, verbose=0).ravel(), 0, CAP); true = np.minimum(load_rul(subset), CAP)
    results[subset] = {"rmse": round(rmse(pred, true), 3), "phm": round(phm_score(pred, true), 1),
                       "coverage": round(float(np.mean(np.abs(pred - true) <= q)), 3)}
    m.save(f"/content/rul_{subset}.keras")
    print(subset, results[subset])

In [ ]:
baseline = json.load(open("data/reports/pdm_cmapss.json")) if os.path.exists("data/reports/pdm_cmapss.json") else {}
for s, r in results.items():
    b = baseline.get(s, {}).get("metrics", {})
    print(f"{s}: deep RMSE {r['rmse']}  vs  gradient-boosted {b.get('test_rmse_capped', 'n/a')} (run scripts/train_public_models.py for the baseline)")
json.dump(results, open("/content/rul_deep_results.json", "w"), indent=1)

In [ ]:
# The backend serves the gradient-boosted RUL models (joblib). Train them here too so VS Code gets the
# exact artifacts + report the API loads (models/pdm_FD00x.joblib, data/reports/pdm_cmapss.json).
!python scripts/train_public_models.py --only pdm
ARTIFACTS = [f"models/pdm_{s}.joblib" for s in ["FD001", "FD002", "FD003", "FD004"]] +             ["data/reports/pdm_cmapss.json", "/content/rul_deep_results.json"] +             [f"/content/rul_{s}.keras" for s in results]

In [ ]:
# Save artifacts to Google Drive (survives the Colab session; avoids flaky browser downloads of big files).
# In VS Code: python scripts/import_models.py <the downloaded aeronexus_export.zip or Drive folder>
import hashlib, json, os, platform, shutil, sys, time
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/aeronexus/export"
os.makedirs(OUT, exist_ok=True)
for item in ARTIFACTS:                                   # files or folders produced above
    dst = os.path.join(OUT, os.path.basename(item.rstrip("/")))
    (shutil.copytree(item, dst, dirs_exist_ok=True) if os.path.isdir(item) else shutil.copy2(item, dst))
def sha(p):
    h = hashlib.sha256(); h.update(open(p, "rb").read()); return h.hexdigest()
import sklearn, numpy
man_path = os.path.join(OUT, "manifest.json")
man = json.load(open(man_path)) if os.path.exists(man_path) else {"artifacts": {}}
for root, _, fs in os.walk(OUT):
    for f in fs:
        if f != "manifest.json":
            p = os.path.join(root, f); man["artifacts"][os.path.relpath(p, OUT)] = sha(p)
man.update({"exported_at": time.strftime("%Y-%m-%dT%H:%M:%S"), "python": platform.python_version(),
            "versions": {"scikit-learn": sklearn.__version__, "numpy": numpy.__version__}})
json.dump(man, open(man_path, "w"), indent=1)
shutil.make_archive("/content/drive/MyDrive/aeronexus/aeronexus_export", "zip", OUT)
print("saved to Drive: MyDrive/aeronexus/aeronexus_export.zip  →", sorted(man["artifacts"])[:12])

**Ship it only if it wins the gate.** Copy `rul_deep_results.json` next to `data/reports/` and compare in the evaluation
page; if the deep model beats the baseline on every subset *and* keeps interval coverage ≥ 0.8, add a Keras loader in
`aeronexus/modules/pdm.py` (`keras.saving.load_model`) — the rest of the workflow (rule, case, approval) is unchanged.